# 🚀 Google Colab Live AI Video API Server (Wan 2.1 / LTX-Video)
Bu notebook, açık kaynaklı **LTX-Video / Wan 2.1** AI video modelini GPU üzerinde canlı bir FastAPI + ngrok web sunucusu olarak çalıştırır. Yerel bilgisayarınızdaki otomasyon sistemi doğrudan bu Colab sunucusuna bağlanarak AI videolar ürettirir.

In [ ]:
# 1. Gerekli Kütüphanelerin Kurulumu
!pip install -q diffusers transformers accelerate torch torchvision imageio-ffmpeg fastapi uvicorn pyngrok nest_asyncio
print('✅ Tüm kütüphaneler kuruldu!')

In [ ]:
# 2. LTX-Video / Wan 2.1 AI Video Modelinin GPU'ya Yüklenmesi
import torch
from diffusers import LTXPipeline
from diffusers.utils import export_to_video

print('🚀 AI Video Modeli Yükleniyor (GPU)...')
pipe = LTXPipeline.from_pretrained('Lightricks/LTX-Video', torch_dtype=torch.bfloat16)
pipe.to('cuda')
print('✅ AI Video Modeli GPU üzerinde hazır!')

In [ ]:
# 3. FastAPI + ngrok Canlı Web Sunucusunun Başlatılması
import os
from fastapi import FastAPI, Response
from pydantic import BaseModel
import uvicorn
import nest_asyncio
from pyngrok import ngrok

app = FastAPI()

class VideoRequest(BaseModel):
    prompt: str
    niche: str = 'minecraft'
    width: int = 576
    height: int = 1024

@app.post('/generate_video')
def generate_video(req: VideoRequest):
    print(f'🎬 Otomasyondan canlı video isteği alındı: {req.prompt}')
    frames = pipe(
        prompt=req.prompt,
        negative_prompt='low quality, blurry, distorted',
        width=req.width,
        height=req.height,
        num_frames=121,
        num_inference_steps=25
    ).frames[0]
    
    out_path = '/content/colab_generated_video.mp4'
    export_to_video(frames, out_path, fps=24)
    
    with open(out_path, 'rb') as f:
        return Response(content=f.read(), media_type='video/mp4')

# Open ngrok tunnel
public_url = ngrok.connect(8000)
print('====================================================')
print('🚀 CANLI GOOGLE COLAB API URL ADRESİNİZ:')
print(public_url)
print('====================================================')
print('👉 Bu adresi bilgisayarınızdaki .env dosyasına ekleyin:')
print(f'COLAB_API_URL={public_url}')
print('====================================================')

nest_asyncio.apply()
uvicorn.run(app, host='0.0.0.0', port=8000)